In [7]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score

from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

2026-05-14 16:47:15.299125: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [8]:
RESULTS_DIR = "/home/jovyan/results"
MODELS_DIR = "/home/jovyan/models"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 5
EPOCHS = 15
LEARNING_RATE = 5e-6

In [9]:
train_df = pd.read_csv(os.path.join(RESULTS_DIR, "train_split_5class.csv"))
val_df = pd.read_csv(os.path.join(RESULTS_DIR, "val_split_5class.csv"))
test_df = pd.read_csv(os.path.join(RESULTS_DIR, "test_split_5class.csv"))

In [10]:
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain distribution:")
print(train_df["dx"].value_counts())

print("\nValidation distribution:")
print(val_df["dx"].value_counts())

print("\nTest distribution:")
print(test_df["dx"].value_counts())

print("\nLabels:")
print(train_df["label"].value_counts().sort_index())

Train: 7025
Validation: 781
Test: 1952

Train distribution:
dx
nv       4827
mel       801
bkl       791
bcc       370
akiec     236
Name: count, dtype: int64

Validation distribution:
dx
nv       537
mel       89
bkl       88
bcc       41
akiec     26
Name: count, dtype: int64

Test distribution:
dx
nv       1341
mel       223
bkl       220
bcc       103
akiec      65
Name: count, dtype: int64

Labels:
label
0     236
1     370
2     791
3     801
4    4827
Name: count, dtype: int64


In [11]:
remove_classes = ["df", "vasc"]

train_df = train_df[~train_df["dx"].isin(remove_classes)]
val_df = val_df[~val_df["dx"].isin(remove_classes)]
test_df = test_df[~test_df["dx"].isin(remove_classes)]

print(len(train_df), len(val_df), len(test_df))

7025 781 1952


In [12]:
label_map = {
    "akiec": 0,
    "bcc": 1,
    "bkl": 2,
    "mel": 3,
    "nv": 4
}

train_df["label"] = train_df["dx"].map(label_map)
val_df["label"] = val_df["dx"].map(label_map)
test_df["label"] = test_df["dx"].map(label_map)

In [13]:
classes = np.sort(train_df["label"].unique())

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"]
)

class_weights = {
    int(c): min(float(w), 3.0)
    for c, w in zip(classes, class_weights_array)}

print(class_weights)

{0: 3.0, 1: 3.0, 2: 1.7762326169405815, 3: 1.7540574282147317, 4: 0.29107105862854776}


In [14]:
def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)

    image = preprocess_input(image)

    return image, label

In [15]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["image_path"].values, train_df["label"].values))

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["image_path"].values, val_df["label"].values))

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_df["image_path"].values, test_df["label"].values))

train_ds = (
    train_ds
    .map(load_image)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE))

val_ds = (
    val_ds
    .map(load_image)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE))

test_ds = (
    test_ds
    .map(load_image)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE))

I0000 00:00:1778777280.830344   46502 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 11977 MB memory:  -> device: 0, name: NVIDIA A16, pci bus id: 0000:46:00.0, compute capability: 8.6


In [16]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

In [17]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

for layer in base_model.layers[-30:]:
    layer.trainable = True

In [18]:
inputs = tf.keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(256, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs, outputs)

In [19]:
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,412,072 (16.83 MB)

 Trainable params: 1,858,149 (7.09 MB)

 Non-trainable params: 2,553,923 (9.74 MB)

In [20]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )
]

In [21]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/15


E0000 00:00:1778777313.299790   46502 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2026-05-14 16:48:45.023618: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900


220/220 ━━━━━━━━━━━━━━━━━━━━ 62s 183ms/step - accuracy: 0.1015 - loss: 2.1167 - val_accuracy: 0.1242 - val_loss: 2.3544
Epoch 2/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 41s 182ms/step - accuracy: 0.1117 - loss: 1.9178 - val_accuracy: 0.1306 - val_loss: 2.6960
Epoch 3/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 42s 184ms/step - accuracy: 0.1274 - loss: 1.7816 - val_accuracy: 0.1472 - val_loss: 2.6409
Epoch 4/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 41s 180ms/step - accuracy: 0.1462 - loss: 1.6966 - val_accuracy: 0.1665 - val_loss: 2.4440
Epoch 5/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 41s 182ms/step - accuracy: 0.1684 - loss: 1.5811 - val_accuracy: 0.2113 - val_loss: 2.2215
Epoch 6/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 44s 193ms/step - accuracy: 0.1916 - loss: 1.5224 - val_accuracy: 0.2586 - val_loss: 2.0131
Epoch 7/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 84s 197ms/step - accuracy: 0.2316 - loss: 1.4481 - val_accuracy: 0.3163 - val_loss: 1.8341
Epoch 8/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 42s 185ms/step - accuracy: 0.2682 - loss: 1.3636 - val

In [24]:
model.save(os.path.join(MODELS_DIR, "efficientnet_5class.keras"))

In [25]:
y_true = np.concatenate([
    y for x, y in test_ds
])

y_probs = model.predict(test_ds)

y_pred = np.argmax(y_probs, axis=1)

2026-05-14 17:03:09.933337: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


61/61 ━━━━━━━━━━━━━━━━━━━━ 7s 111ms/step


In [26]:
cm = confusion_matrix(y_true, y_pred)

print(cm)

print("\nClassification Report:\n")

target_names = list(label_map.keys())

print(classification_report(
    y_true,
    y_pred,
    target_names=target_names
))

balanced_acc = balanced_accuracy_score(y_true, y_pred)

print("\nBalanced Accuracy:", balanced_acc)

[[ 25   8  19  12   1]
 [  8  56  22  15   2]
 [  9  17 152  30  12]
 [ 13   8  57 120  25]
 [ 19  53 178 292 799]]

Classification Report:

              precision    recall  f1-score   support

       akiec       0.34      0.38      0.36        65
         bcc       0.39      0.54      0.46       103
         bkl       0.36      0.69      0.47       220
         mel       0.26      0.54      0.35       223
          nv       0.95      0.60      0.73      1341

    accuracy                           0.59      1952
   macro avg       0.46      0.55      0.47      1952
weighted avg       0.76      0.59      0.63      1952


Balanced Accuracy: 0.5506308799544941


In [37]:
import numpy as np

cm = np.array([
 [ 25,   8,  19,  12,   1]
 [  8,  56,  22,  15,   2]
 [  9,  17, 152,  30,  12]
 [ 13,   8,  57, 120,  25]
 [ 19,  53, 178, 292, 799]
])

class_index = 3  # melanoma

tp = cm[class_index, class_index]
fn = np.sum(cm[class_index, :]) - tp
fp = np.sum(cm[:, class_index]) - tp
tn = np.sum(cm) - (tp + fn + fp)

specificity = tn / (tn + fp)

print("Melanoma Specificity:", specificity)

<>:4: SyntaxWarning: list indices must be integers or slices, not tuple; perhaps you missed a comma?
<>:4: SyntaxWarning: list indices must be integers or slices, not tuple; perhaps you missed a comma?
/tmp/ipykernel_46502/3433711733.py:4: SyntaxWarning: list indices must be integers or slices, not tuple; perhaps you missed a comma?
  [ 25,   8,  19,  12,   1]


TypeError: list indices must be integers or slices, not tuple

In [34]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

cm = np.array([
[ 25,   8,  19,  12   1]
 [  8  56  22  15   2]
 [  9  17 152  30  12]
 [ 13   8  57 120  25]
 [ 19  53 178 292 799]]
])

class_names = ["akiec", "bcc", "bkl", "mel", "nv"]

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("EfficientNetB0 Confusion Matrix")

plt.tight_layout()

plt.savefig("/home/jovyan/results/efficientnet_confusion_matrix.png")

plt.show()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2362546291.py, line 6)

In [29]:
print(len(train_df))
print(len(val_df))
print(len(test_df))

print(train_df["label"].value_counts())
print(test_df["label"].value_counts())

7025
781
1952
label
4    4827
3     801
2     791
1     370
0     236
Name: count, dtype: int64
label
4    1341
3     223
2     220
1     103
0      65
Name: count, dtype: int64
